# Reference mapping onto the autoimmune CD4+ T / B cell atlases (Python, symphonypy)

Yasumizu et al., *Cell Genomics* 2024 (CD4+ T cells) and the companion B cell study.
This notebook maps a **raw-count query** of one lineage onto the matching reference atlas and
transfers `cluster_L1` / `cluster_L2` labels and the published UMAP coordinates.

## Read this before you run it

Both atlases were built as

```
normalize_total(target_sum=1e4) -> log1p
  -> regress_out([total_counts, pct_counts_mt, S_score, G2M_score]) -> scale -> PCA -> Harmony
```

so `reference.var['mean']` and `reference.var['std']` are statistics of the **regression
residuals**. `symphonypy` scales the query with those statistics, so feeding it plain
`log1p` counts offsets the projection. The query must therefore be residualised with the
**reference's own regression coefficients** (`reference.varm['regress_beta']`), and the
reference must carry a real Symphony compression object (`reference.uns['harmony']`) or
`symphonypy` silently falls back to a soft k-means cache.

Self-mapping of reference cells (kNN-5 `cluster_L2` agreement, 6,000 cells):

| query preprocessing | reference | B cells | CD4+ T cells |
|---|---|---|---|
| stored embedding (ceiling) | - | 87.1% | 81.6% |
| `log1p` only | no `uns['harmony']` | 57.7% | 49.1% |
| `log1p` only, CP100K | no `uns['harmony']` | 71.0% | - |
| residuals | no `uns['harmony']` | 55.9% | - |
| **residuals (this notebook)** | **`uns['harmony']`, `key='sample'`** | **84.8%** | **77.5%** |
| residuals | `uns['harmony']`, `key=None` | 81.4% | 72.6% |

Reference PCA loadings, Harmony embedding, UMAP and labels are never modified: the published
coordinates of the papers are reproduced exactly.

## 1. Inputs

* `REFERENCE_H5AD` - the distributed reference (`...symphony_*.h5ad`).
* `QUERY_H5AD` - your cells, **raw counts** in `X`, human gene symbols in `var_names`,
  already restricted to the lineage of the reference. Use Azimuth / CellTypist upstream to
  isolate B cells or CD4+ T cells (see the R pipeline in this repository for the Azimuth step).
* `BATCH_KEY` - column in `query.obs` naming the library/batch. Set to `None` for a single
  library; that costs a few points of concordance (see the table above).

In [ ]:
from pathlib import Path

REFERENCE_H5AD = 'autoimmune_bcell_symphony_reference_v2.h5ad'
QUERY_H5AD = 'query_raw_counts.h5ad'
CELL_CYCLE_GENES = 'regev_lab_cell_cycle_genes.txt'

BATCH_KEY = 'sample'          # or None
SYMBOL_COLUMN = None          # e.g. 'gene_symbol' if var_names are Ensembl IDs
OUT_PREFIX = Path('output/query_mapped')

OUT_PREFIX.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import symphonypy as sp

import screfmapping_symphonypy as srm

np.random.seed(0)
sc.settings.verbosity = 1
print('scanpy', sc.__version__, '| symphonypy', sp.__version__)

## 2. Load the reference and check that it is Symphony-ready

In [ ]:
reference = sc.read_h5ad(REFERENCE_H5AD)
print(reference)

assert 'regress_beta' in reference.varm, 'reference is missing varm["regress_beta"]'
assert 'harmony' in reference.uns, 'reference is missing uns["harmony"] (Symphony compression)'

recipe = reference.uns.get('symphony_query_preprocessing', {})
for step in recipe.get('steps', []):
    print('-', step)

## 3. Preprocess the query exactly like the reference

`srm.prepare_query` runs steps 1-6 of the recipe and returns the residual matrix on the
reference gene set. Genes absent from the query are left at the reference mean (0 after
residualisation), so a query with fewer genes still maps, with proportionally more noise.

In [ ]:
query_raw = sc.read_h5ad(QUERY_H5AD)
print(query_raw)

query = srm.prepare_query(
    query_raw,
    reference,
    cell_cycle_genes=CELL_CYCLE_GENES,
    batch_key=BATCH_KEY,
    symbol_column=SYMBOL_COLUMN,
)
query

### Sanity check: are the query covariates on the same scale as the reference?

The regression coefficients are only transferable if `total_counts`, `pct_counts_mt`,
`S_score` and `G2M_score` live on the same scale in both datasets. A query whose
`total_counts` are an order of magnitude away from the reference (very shallow or very deep
libraries) will be over- or under-corrected; inspect the mapping quality below before
trusting the labels.

In [ ]:
summary = query.obs[srm.COVARIATES].describe().loc[['mean', '50%', 'std']]
ref_summary = recipe.get('reference_covariate_summary')
if ref_summary:
    ref_summary = pd.DataFrame(ref_summary)[srm.COVARIATES]
    print('reference:'); display(ref_summary.round(3))
print('query:'); display(summary.round(3))

## 4. Map, transfer labels, project onto the published UMAP

In [ ]:
query = srm.map_query(
    query,
    reference,
    key='sample' if BATCH_KEY else None,
    labels=('cluster_L1', 'cluster_L2'),
    n_neighbors=5,
    ingest=True,
)

print(query.obs['cluster_L2'].value_counts().to_string())
print()
print('symphony_dist (per-cell confidence, higher = less confident):')
print(query.obs['symphony_dist'].describe().round(3).to_string())

## 5. Look at the result

Query cells on the published reference UMAP. Cells that land outside the reference manifold
(high `symphony_dist`) are the ones to distrust - typically doublets, a different lineage
leaking in from the upstream extraction step, or very low-depth cells.

In [ ]:
import matplotlib.pyplot as plt

ref_umap = np.asarray(reference.obsm['X_umap'])
q_umap = np.asarray(query.obsm['X_umap'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax in axes:
    ax.scatter(ref_umap[:, 0], ref_umap[:, 1], s=1, c='lightgrey', linewidths=0, rasterized=True)
    ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')

for label in query.obs['cluster_L2'].astype(str).unique():
    m = (query.obs['cluster_L2'].astype(str) == label).to_numpy()
    axes[0].scatter(q_umap[m, 0], q_umap[m, 1], s=4, linewidths=0, label=label, rasterized=True)
axes[0].set_title('query, transferred cluster_L2')
axes[0].legend(markerscale=3, fontsize=7, loc='center left', bbox_to_anchor=(1.0, 0.5))

sc_ = axes[1].scatter(q_umap[:, 0], q_umap[:, 1], s=4, c=query.obs['symphony_dist'],
                      cmap='viridis', linewidths=0, rasterized=True)
axes[1].set_title('query, symphony_dist')
fig.colorbar(sc_, ax=axes[1], shrink=0.8)
fig.tight_layout()
fig.savefig(f'{OUT_PREFIX}_umap.pdf', bbox_inches='tight')
plt.show()

In [ ]:
cols = ['cluster_L1', 'cluster_L2', 'symphony_dist']
if BATCH_KEY:
    cols = ['sample'] + cols
obs_out = query.obs[cols].copy()
obs_out[['umap_x', 'umap_y']] = q_umap
obs_out.to_csv(f'{OUT_PREFIX}_labels.tsv.gz', sep='\t')
query.write_h5ad(f'{OUT_PREFIX}.h5ad', compression='gzip')
print('wrote', f'{OUT_PREFIX}_labels.tsv.gz', 'and', f'{OUT_PREFIX}.h5ad')

## 6. Composition per batch

Useful first read-out and a quick sanity check: a batch whose composition is wildly
different from the others is usually an upstream extraction problem, not biology.

In [ ]:
if BATCH_KEY:
    ct = pd.crosstab(query.obs['sample'], query.obs['cluster_L2'])
    display((ct.div(ct.sum(axis=1), axis=0) * 100).round(1))
else:
    display(query.obs['cluster_L2'].value_counts(normalize=True).mul(100).round(1))

## Troubleshooting

**Do not** use the older instruction `normalize_total(target_sum=1e5) -> log1p ->
map_embedding`. It was written before the residual scaling was understood; CP100K happened
to compensate for part of the offset empirically (71% instead of 58% self-agreement) but it
is not the reference recipe. Use CP10K plus residualisation.

| symptom | cause |
|---|---|
| `KeyError: regress_beta` | reference predates the fix; use the `symphony_*` release |
| `KeyError: harmony` | same; a reference without `uns['harmony']` loses ~30 points |
| all cells collapse onto one label | query was not residualised, or `total_counts` is on a different scale |
| `symphony_dist` uniformly high | query contains cells of another lineage, or too few reference genes were found |
| labels differ from a previous run | `key=` changed, or the reference lacked `uns['harmony']` and used a random soft k-means cache |

Mapping is deterministic given the same reference, `key` and `n_neighbors`; set
`np.random.seed` before `map_query` if you compare runs.

## Citation

Yasumizu, Y. et al. Single-cell transcriptome landscape of circulating CD4+ T cell
populations in autoimmune diseases. *Cell Genomics* (2024).
https://doi.org/10.1016/j.xgen.2023.100473